# Journal de bord – Projet CISIA : prédiction de la classe de retour à l'emploi

## Sommaire

| Phase | Jours | Sections | Livrable clé |
|---|---|---|---|
| Cadrage et acquisition | [J1](#jour-1) – [J2](#jour-2) | §1–2 | Cadrage métier, dictionnaire de variables |
| EDA et préparation | [J3](#jour-3) – [J5](#jour-5) | §3–4 | Diagnostic qualité et biais, pipeline reproductible, assertions de qualité |
| Modélisation et arbitrage | [J6](#jour-6) – [J8](#jour-8) | §5–6 | 16 baselines, GridSearchCV corrigé, choix du scénario `multimodal_ethique` |
| Explicabilité, équité, abstention | [J9](#jour-9) – [J10](#jour-10) | §7 | SHAP, audit par sous-groupe, seuil de rejet calibré |
| Industrialisation et suivi | [J11](#jour-11) – [J15](#jour-15) | §8–9 | API, feedback, drift, calibration, gate de release, retraining |
| Revue et durcissement | [J16](#jour-16) | §8–9 | Cohérence de la documentation, plan d'action |
| Annexes | [A1](#annexes) | – | Décisions, critères de succès, hypothèses, écarts, limites, soutenance |

<a id="jour-1"></a>
## 📅 Journal de bord – Jour 1 (25/08/2026)

### 🔧 Étapes / Actions réalisées
- Prise de connaissance du sujet de certification et du jeu de données `dataset_trajectoire_emploi.csv`.
- Rédaction du cadrage métier (§1) : identification du client (agence pour l'emploi), du besoin (aiguillage des demandeurs d'emploi vers un accompagnement adapté) et des bénéficiaires directs (conseillers) et indirects (usagers).
- Reformulation du problème en tâche IA : classification multiclasses (3 classes) visant à prédire `classe_retour_emploi` (retour < 6 mois / 6-12 mois / > 12 mois).
- Première rédaction du tableau de critères de succès (cible initiale côté client), en distinguant critère métier, critère modèle et critère opérationnel.
- Repérage à chaud des premiers risques éthiques et réglementaires pressentis : présence de variables potentiellement sensibles (`age`, `nationalite_hors_ue`) et cadre RGPD à respecter sur les données individuelles d'usagers.

### 📝 Observations / Difficultés
- La reformulation en tâche de classification multiclasses (3 classes) n'a pas posé de difficulté : la cible `classe_retour_emploi` est clairement définie. 
- Identification des proxys : repérer les variables sensibles directes (`age`, `nationalite_hors_ue`) était simple, mais identifier les variables pouvant les reconstituer indirectement (`code_rome_vise`, `code_insee_commune`, `synthese_entretien`) l'était beaucoup moins. Ce point a été noté comme risque à traiter en EDA puis en audit de biais, et non comme un problème résolu.
- Connaissance du cadre éthique et réglementaire : le cadre applicable (RGPD, minimisation des données, discrimination indirecte, critères prohibés par le droit français) n'était pas maîtrisé d'emblée. Une montée en compétence est nécessaire avant de décider quelles variables conserver.


### ✅ Prochaines étapes
- Finaliser la synthèse de cadrage (§1.6).
- Identifier précisément la source, le format et la volumétrie du jeu de données (§2.1).
- Charger le dataset et produire un premier dictionnaire de variables (§2.2-2.3).
- Repérer dès le chargement les variables à risque.

---

<a id="jour-2"></a>
## 📅 Journal de bord – Jour 2 (26/08/2026)

### 🔧 Étapes / Actions réalisées
- Documentation de la source des données (§2.1) : CSV de l'agence nationale de l'emploi, ~2 500 lignes × 10 colonnes, accès en lecture seule, date d'extraction non communiquée.
- Chargement du dataset (§2.2) et premier contrôle de la structure (types de colonnes, nombre de lignes, aperçu).
- Construction du dictionnaire de variables (§2.3) : nature (numérique / catégorielle / texte), plage de valeurs, et statut « cible » / « sensible » pour chacune des 10 variables (`usager_id`, `age`, `niveau_diplome`, `anciennete_poste_ans`, `code_rome_vise`, `code_insee_commune`, `est_allocataire`, `nationalite_hors_ue`, `synthese_entretien`, `departement`, `classe_retour_emploi`).
- Rédaction de la synthèse d'acquisition (§2.4) : origine administrative mixte (données déclaratives + saisies conseiller), volumétrie modeste à mettre en regard des 3 classes cibles, et premiers signaux de vigilance sur `code_insee_commune` (quasi-identifiant, 2407 modalités pour 2500 lignes) et sur `synthese_entretien` (longueur de texte très resserrée, 63-77 caractères).

### 📝 Observations / Difficultés
- `code_insee_commune` a un nombre de modalités quasi égal au nombre de lignes : cela le rend inutilisable tel quel pour la modélisation et pose la question d'un regroupement (ex. département) à explorer en EDA.
- La variable texte `synthese_entretien` interroge dès la lecture : sa faible variabilité de longueur suggère un champ pré-rédigé ou semi-automatisé plutôt qu'une note libre — hypothèse à vérifier lors de l'analyse textuelle (§3.3.3).

### ✅ Prochaines étapes
- Démarrer l'EDA (§3) : premier coup d'œil global, puis contrôle qualité (manquants, doublons, valeurs aberrantes).
- Étudier la distribution des variables numériques, catégorielles et de la variable texte.
- Examiner l'équilibre de la variable cible `classe_retour_emploi`.

---

<a id="jour-3"></a>
## 📅 Journal de bord – Jour 3 (27/08/2026)

### 🔧 Étapes / Actions réalisées
- Premier coup d'œil global sur le dataset (§3.1) : types, statistiques descriptives, valeurs uniques par colonne.
- Contrôle qualité (§3.2) : détection des valeurs manquantes (`age` ~4,9 %, `niveau_diplome` ~3,4 %, `est_allocataire` ~1,8 %, `synthese_entretien` ~3,2 %) et des doublons (aucun doublon exact détecté, ni global ni sur `usager_id`).
- Détection des valeurs aberrantes par méthode IQR sur les variables numériques : aucun outlier sur `age`, mais 126 observations identifiées comme extrêmes sur `anciennete_poste_ans` (au-delà de ~9 ans, jusqu'à 22,6 ans).
- Analyse de la distribution des variables numériques (§3.3.1) : `age` quasi uniforme entre 18 et 63 ans, `anciennete_poste_ans` fortement asymétrique à droite.
- Analyse des variables catégorielles (§3.3.2) : fort déséquilibre sur `nationalite_hors_ue` (88,2 % / 11,8 %), répartition plus équilibrée sur `est_allocataire`, modalité « Bac » dominante sur `niveau_diplome`, `code_rome_vise` à 50 modalités regroupables en 12 familles professionnelles.
- Première exploration de la variable textuelle `synthese_entretien` (§3.3.3) et des regroupements potentiels (§3.3.4, ex. `code_insee_commune` → `departement`).

### 📝 Observations / Difficultés
- Les outliers d'`anciennete_poste_ans` ne sont pas supprimés à ce stade : ils peuvent correspondre à des situations réelles (reconversion tardive, longue carrière) et devront être examinés au regard des contraintes métier plutôt qu'écartés mécaniquement.
- Le déséquilibre marqué de `nationalite_hors_ue` (88,2 %/11,8 %) est noté comme point de vigilance pour l'analyse de biais à venir : le sous-groupe minoritaire sera statistiquement plus fragile.
- Les taux de valeurs manquantes restent modérés (< 5 % par variable) et ne remettent pas en cause l'exploitabilité du dataset, mais imposent de choisir une stratégie d'imputation explicite en §4 plutôt que de supprimer les lignes concernées.

### ✅ Prochaines étapes
- Étudier l'équilibre de la variable cible (§3.4).
- Analyser les corrélations et croisements entre variables et cible (§3.5).
- Mener l'analyse des biais et variables sensibles (§3.6) : disparate impact, biais lexical potentiel dans `synthese_entretien`.
- Rédiger la synthèse EDA (§3.7) et réviser le tableau de critères de succès à la lumière des observations.

---

<a id="jour-4"></a>
## 📅 Journal de bord – Jour 4 (28/08/2026)

### 🔧 Étapes / Actions réalisées
- Analyse de l'équilibre de la cible (§3.4) : classe 1 (retour 6-12 mois) 44,48 %, classe 0 (retour < 6 mois) 37,44 %, classe 2 (retour > 12 mois, minoritaire) 18,08 %.
- Étude des corrélations et croisements (§3.5) : matrice de corrélation, pairplot numériques par classe, moyennes par classe pour les variables numériques, tests du χ² pour les variables catégorielles. `age` ressort comme la variable numérique la plus discriminante (moyenne croissante de 36,8 à 46,5 ans selon la classe) ; `anciennete_poste_ans` apporte peu de signal (pas de colinéarité, r = -0,015). `synthese_entretien`, `niveau_diplome` et `code_rome_vise` présentent les associations les plus fortes avec la cible.
- Analyse des biais et variables sensibles (§3.6) : heatmaps de croisement cible / `nationalite_hors_ue`, `groupe_age`, `niveau_diplome` ; recherche de mentions explicites de critères sensibles dans `synthese_entretien` (âge, origine, genre) ; test de prédictibilité d'une variable sensible à partir du texte (signal de biais lexical implicite) ; calcul du disparate impact (règle des 4/5) sur `nationalite_hors_ue`, `groupe_age`, `est_allocataire`, `niveau_diplome`.
- Rédaction de la synthèse EDA (§3.7) et révision du tableau de critères de succès : ajout du taux d'erreur critique de sous-classement (Classe 2 → Classe 0 < 5 %), du recall Classe 2 (≥ 0,85) et du F1-macro (≥ 0,75) comme métriques de pilotage, articulées entre critère métier, critère technique et critère de cohérence globale.

### 📝 Observations / Difficultés
- Le déséquilibre modéré de la cible confirme qu'une accuracy seule serait trompeuse (un modèle qui prédirait systématiquement la classe 1 atteindrait déjà ~44 % de bonnes classifications) ; le choix du F1-macro et du recall de la classe 2 comme métriques de pilotage a été argumenté sur cette base.
- La traduction du besoin métier (éviter le sous-classement des usagers à risque de longue durée) en une métrique technique précise (recall Classe 2 ≥ 0,85) a demandé plusieurs itérations pour rester reliée à un seuil d'erreur tolérable côté métier plutôt qu'à un objectif arbitraire.
- Le test de biais lexical sur `synthese_entretien` a nécessité une vectorisation TF-IDF préliminaire pour vérifier si le texte permettait de prédire indirectement une variable sensible : ce point a été traité comme un signal à surveiller en modélisation plutôt que comme un motif d'exclusion immédiate de la variable.
- Choix de la représentation du texte `synthese_entretien` : cette variable est à la fois la plus riche en signal  et la plus difficile à encoder. Trois options ont été comparées.

  | Option | Avantage | Limite retenue |
  |---|---|---|
  | One-Hot sur la valeur complète | Simple, lisible | Fige les valeurs connues à l'entraînement ; une formulation inédite n'a aucune colonne et devient une ligne de zéros. Ne capte aucune proximité entre deux phrases proches. |
  | **TF-IDF** | Pondère les mots discriminants, produit un vecteur pour tout texte partageant du vocabulaire avec le corpus, reste utilisable si les conseillers reformulent | Voir limites ci-dessous. |

- **Pourquoi le TF-IDF a été préféré** : le texte est susceptible d'évoluer en production (nouvelles formulations, changement de conseillers ou de gabarit). Le TF-IDF généralise au niveau des mots et non de la phrase entière. Il s'intègre aussi dans le `Pipeline` scikit-learn : le vocabulaire et les pondérations IDF sont appris sur le train uniquement, ce qui évite toute fuite vers le test.

- **Hypothèse de départ à vérifier** : en J2, la longueur des textes était très resserrée (63 à 77 caractères), ce qui suggère un champ pré-rédigé ou semi-automatisé. Si le nombre de phrases distinctes est faible, un One-Hot sur la phrase entière aurait été défendable à court terme. Le nombre de valeurs uniques doit donc être mesuré et consigné, car il conditionne la pertinence du choix.

- **Limites identifiées du TF-IDF** :
  - *Vocabulaire figé* : un mot absent du corpus d'entraînement est ignoré sans alerte. Le TF-IDF ne supprime donc pas le problème de l'évolution du texte, il l'atténue.
  - *Ordre et négation* : en unigrammes, « difficulté de mobilité » et « pas de difficulté de mobilité » donnent presque le même vecteur. Les bigrammes atténuent ce défaut mais augmentent la dimension.
  - *Dimension élevée et parcimonie* : avec ~2 500 lignes, un grand vocabulaire favorise le surapprentissage. Les réglages `min_df`, `max_features` et `ngram_range` doivent être justifiés (valeurs retenues : à confirmer).
  - *Interprétabilité* : les colonnes sont des mots, ce qui permet SHAP (J9) et la lecture métier des termes influents.

- **Risque éthique propre au texte** : le TF-IDF donne au modèle accès à tout mot présent, y compris ceux qui reconstituent indirectement une variable sensible (âge, origine, genre). Le test de prédictibilité de J4 (TF-IDF vers variable sensible) est donc indissociable de ce choix. Le texte a été conservé dans le scénario éthique parce que SHAP (J9) montre que les mots influents portent sur les difficultés cumulées et les freins périphériques, mais l'audit doit être maintenu.

- **Validation du choix** : le scénario `multimodal_complet` domine `tabulaire_seul` et `texte_seul` (J6), ce qui confirme l'apport du texte en complément du tabulaire. Une

### ✅ Prochaines étapes
- Réaliser le split train/test (§4.1), stratifié sur la cible.
- Construire le pipeline de preprocessing (§4.2) : imputation et standardisation des variables numériques, imputation et encodage des variables catégorielles, vectorisation TF-IDF de `synthese_entretien`.
- Définir les scénarios de jeux de données (§4.3) permettant de comparer performance maximale, retrait des variables sensibles, et contribution du texte.
- Rédiger la synthèse de préparation (§4.4) et les assertions de qualité (§4.5) avant de passer à la modélisation.

---

<a id="jour-5"></a>
## 📅 Journal de bord – Jour 5 (29/08/2026)

### 🔧 Étapes / Actions réalisées
- Réalisation du split train/test (§4.1) avec stratification sur `classe_retour_emploi` pour préserver la répartition des 3 classes dans les deux jeux, `random_state=42` fixé pour la reproductibilité.
- Construction du pipeline de preprocessing (§4.2) avec un `ColumnTransformer` intégré à un `Pipeline` scikit-learn, afin d'éviter toute fuite de données entre prétraitement et modélisation : imputation puis standardisation (`StandardScaler`) pour `age` et `anciennete_poste_ans` ; imputation puis encodage (`OneHotEncoder`) pour `niveau_diplome`, `code_rome_vise`, `departement`, `est_allocataire`, `nationalite_hors_ue` ; vectorisation `TfidfVectorizer` pour `synthese_entretien`.
- Définition des scénarios de jeux de données (§4.3), dont le scénario S1 « Multimodal complet » (toutes les variables exploitables hors `usager_id`) servant de référence pour mesurer la performance maximale atteignable avant atténuation des risques de biais, et un scénario retirant les variables sensibles directes pour en mesurer l'impact.
- Rédaction de la synthèse de préparation (§4.4), justifiant en particulier le choix du TF-IDF plutôt que du One-Hot sur `synthese_entretien` : le texte libre est susceptible d'évoluer (nouvelles formulations des conseillers), et un encodage figé au moment de l'entraînement ne saurait pas traiter une valeur textuelle inédite, contrairement au TF-IDF qui produit un vecteur cohérent pour tout texte partageant du vocabulaire avec le corpus connu.
- Mise en place des assertions de qualité (§4.5) sur les données préparées (absence de manquants résiduels dans la cible, absence de doublons résiduels, cohérence de la taille des splits) et premier lancement des tests associés via `pytest`.

### 📝 Observations / Difficultés
- Le choix de la stratégie d'encodage pour `synthese_entretien` a nécessité une réflexion approfondie : le TF-IDF a été préféré au One-Hot car il ne fige pas les catégories connues au moment de l'entraînement et reste utilisable si de nouvelles formulations apparaissent en production, ce qui rejoint une préoccupation d'industrialisation abordée plus tard dans le projet.
- La stratification du split sur une cible à 3 classes déséquilibrées a demandé une vérification explicite que les proportions de classes restaient comparables entre train et test, faute de quoi les métriques de la classe 2 (minoritaire) auraient pu être biaisées dès l'évaluation.
- Les assertions de qualité (§4.5), volontairement conçues pour arrêter le notebook en cas d'échec, ont été traitées comme première brique de tests automatisés du projet plutôt que comme une simple vérification manuelle — logique qui sera généralisée plus tard avec `pytest`.

### ✅ Bilan et prochaines étapes
- Le cadrage métier, l'acquisition des données, l'EDA et la préparation des données (parties 1 à 4) sont finalisés : le jeu de données est nettoyé, compris, et transformé en jeux d'entraînement/test exploitables via un pipeline reproductible.
- Les critères de succès ont été révisés à la lumière de l'EDA (recall Classe 2 ≥ 0,85, F1-macro ≥ 0,75) et guideront le choix et l'évaluation des modèles.
- Prochaine étape : passer à la modélisation & entraînement (§5) — sélection des familles de modèles candidates, validation croisée, comparaison des scénarios de jeux de données définis en §4.3.

---

<a id="jour-6"></a>
## 📅 Journal de bord – Jour 6 (30/08/2026)

### 🔧 Étapes / Actions réalisées
- Démarrage de la modélisation (§5) : définition du protocole d'évaluation — holdout stratifié de 20 % réservé exclusivement à l'évaluation finale, validation croisée stratifiée (5 folds) réalisée uniquement sur le train.
- Choix des métriques de pilotage de la validation croisée (§5.2) : F1-macro (vue d'ensemble équilibrée), recall Classe 2 et F1 Classe 2 (détection de la classe à risque), taux d'erreur critique 2→0 (erreur la plus coûteuse côté métier). L'accuracy est conservée mais reléguée en indicateur secondaire, le déséquilibre de la cible pouvant la rendre trompeuse.
- Entraînement d'une première série de baselines (§5.3-5.4) : 4 algorithmes (régression logistique, random forest, LightGBM, XGBoost) croisés avec les 4 scénarios de jeux de données (§4.3), en configuration `balanced` (pondération de classe / poids d'échantillon).
- Construction d'un tableau de synthèse comparant les 16 combinaisons sur F1-macro, recall Classe 2 et taux d'erreur critique 2→0 (moyenne et écart-type sur les folds).

### 📝 Observations / Difficultés
- Le scénario `multimodal_complet` domine largement les scénarios mono-modalité (`texte_seul`, `tabulaire_seul`) sur toutes les métriques dès cette première série de baselines, confirmant l'intérêt de la double modalité texte + tabulaire pressenti en EDA.
- Les résultats obtenus avec la seule configuration `balanced` ne permettent pas encore de distinguer clairement le meilleur algorithme par scénario : un travail de recherche d'hyperparamètres ciblé sur la classe 2 est nécessaire avant de pouvoir arbitrer (prévu §5.5).
- Point de vigilance noté pour la suite : le critère de sélection utilisé par défaut dans le code de recherche d'hyperparamètres priorise le F1-macro avant les métriques spécifiques à la classe 2 — à surveiller de près une fois le GridSearchCV mis en place, car cet ordre de priorité ne reflète pas nécessairement l'objectif métier prioritaire.

### ✅ Prochaines étapes
- Mettre en place une recherche d'hyperparamètres (GridSearchCV) dédiée à la classe 2 pour chacun des 4 algorithmes (§5.5).
- Vérifier concrètement si le critère de sélection du GridSearchCV (ordre de priorité des métriques) correspond bien à l'objectif métier, et non uniquement à la performance globale.

---

<a id="jour-7"></a>
## 📅 Journal de bord – Jour 7 (31/08/2026)

### 🔧 Étapes / Actions réalisées
- Implémentation de `GridSearchCV` dédiés à la classe 2 pour les 4 algorithmes (§5.5), avec des scorers personnalisés : F1-macro, F1 Classe 2, recall Classe 2, taux d'erreur critique 2→0 — chacun calculé à chaque fold via une fonction de scoring dédiée (`make_scorer`).
- Premier jeu de résultats obtenu et configurations `best_class_2_*` retenues pour chaque couple algorithme/scénario, sur la base du critère de sélection existant (priorité : F1-macro, puis F1 Classe 2, puis erreur critique 2→0).
- Calcul, en complément, d'un second taux d'erreur asymétrique jusque-là non mesuré : le taux 0→2 (usager sans risque classé à tort comme prioritaire), pour disposer d'une vision complète des deux types d'erreur possibles sur la classe 2.

### 📝 Observations / Difficultés
- **Découverte importante** : la configuration retenue par le GridSearchCV pour XGBoost sur le scénario `multimodal_ethique` (`best_class_2_ethique`) présente un taux d'erreur 0→2 très élevé (28 cas sur le jeu de calibration, contre seulement 6 à 13 pour les autres configurations) — le modèle sur-classe massivement des usagers sans risque comme prioritaires. Ce défaut n'était pas visible tant que seul le taux 2→0 était surveillé.
- Analyse de la cause : le critère de sélection du GridSearchCV (`_select_best_grid_search_candidate`) ne prenait en compte que le F1-macro, le F1 Classe 2 et le taux 2→0 — **le taux 0→2 n'entrait jamais dans le calcul de sélection**, alors même qu'il était déjà calculable. Le GridSearchCV pouvait donc choisir un jeu d'hyperparamètres minimisant l'erreur grave en laissant l'autre erreur dériver sans contrainte.
- **Correction apportée** : ajout d'un scorer dédié `error_0_to_2`, puis remplacement du critère de sélection par un score de coût métier combiné (`3 × erreur_2→0 + 1 × erreur_0→2`), le ratio de pondération 3:1 traduisant l'hypothèse qu'une erreur grave coûte davantage qu'une fausse alerte. Ce ratio est documenté comme hypothèse de travail à valider avec un porteur métier, et non comme une valeur validée empiriquement.
- Les 4 GridSearchCV ont été relancés avec le critère corrigé ; les nouvelles configurations `best_class_2_*` diffèrent sensiblement des précédentes sur certains hyperparamètres (ex. `n_estimators`, `learning_rate` pour LightGBM), confirmant que le critère initial orientait réellement la sélection vers un optimum différent.

### ✅ Prochaines étapes
- Relancer l'évaluation finale sur le holdout (§5.6) avec les configurations issues du GridSearchCV corrigé.
- Construire le tableau de comparaison des scénarios (§6.1) en intégrant explicitement les deux taux d'erreur (2→0 et 0→2), pas seulement le premier.

---

<a id="jour-8"></a>
## 📅 Journal de bord – Jour 8 (01/09/2026)

### 🔧 Étapes / Actions réalisées
- Évaluation finale sur le holdout (§5.6) de l'ensemble des combinaisons modèle/scénario/configuration retenues, avec calcul systématique de : accuracy, F1-macro, recall/précision/F1 Classe 2, taux d'erreur 2→0 et 0→2, ROC AUC macro, ainsi que deux indicateurs de coût opérationnel (latence p95, CPU par 1000 prédictions) issus d'un benchmark dédié.
- Construction du tableau comparatif des 4 scénarios (§6.1), avec sélection de la meilleure configuration par scénario selon le score de coût combiné défini la veille.
- Rédaction de la comparaison argumentée performance / éthique / robustesse (§6) et de la recommandation finale (§6.2-6.3).

### 📝 Observations / Difficultés
- Les mesures de latence et de coût CPU se sont révélées instables d'une exécution à l'autre pour une même combinaison (écarts jusqu'à ×3 observés sur XGBoost `multimodal_complet` entre deux runs), probablement liés à la charge de la machine au moment du test plutôt qu'à une variation réelle du modèle. Ce point est documenté comme limite : ces indicateurs de coût sont traités comme des ordres de grandeur relatifs entre modèles, pas comme des valeurs absolues fiables sans répétition des mesures.
- Arbitrage central de la journée : le scénario `multimodal_complet` reste supérieur sur toutes les métriques de performance brute, mais s'appuie sur la nationalité hors UE et l'âge — deux caractéristiques directement associées à des critères de discrimination prohibés par le droit français (article 225-1 du Code pénal). Le scénario `multimodal_ethique`, qui les retire, ne dégrade que marginalement le taux d'erreur critique 2→0 (+1 cas sur l'ensemble du holdout) mais multiplie par un facteur notable le taux de fausses alertes 0→2.
- Décision retenue : privilégier le scénario `multimodal_ethique` malgré son coût de performance globale, le risque juridique et éthique d'une variable directement discriminante étant jugé plus grave qu'une charge opérationnelle accrue liée aux fausses alertes — ce compromis est explicitement assumé et chiffré dans la synthèse plutôt que présenté comme un choix neutre.

### ✅ Prochaines étapes
- Approfondir l'explicabilité du modèle retenu (§7.1, SHAP) pour vérifier concrètement l'influence relative des différentes variables, en particulier l'absence d'effet direct de la nationalité.
- Auditer l'existence d'un biais résiduel par sous-groupe (§7.2), pour vérifier si le retrait des variables sensibles suffit à éliminer les écarts de traitement ou si un effet indirect persiste via des variables corrélées.

---

<a id="jour-9"></a>
## 📅 Journal de bord – Jour 9 (02/09/2026)

### 🔧 Étapes / Actions réalisées
- Analyse d'explicabilité SHAP (§7.1) sur le modèle retenu, avec un script dédié restreignant l'usage de `TreeExplainer` aux modèles à base d'arbres et calculant les contributions spécifiquement pour la classe 2.
- Audit de biais par sous-groupe (§7.2.1) : reconstruction des matrices de confusion sur le holdout, ventilées par tranche d'âge, nationalité, statut allocataire et niveau de diplôme, avec un seuil de masquage (effectif < 20 cas de la classe concernée) pour éviter d'interpréter des taux statistiquement instables.

### 📝 Observations / Difficultés
- L'analyse SHAP confirme que la nationalité (déjà retirée du modèle) et l'âge ne figurent pas parmi les facteurs les plus influents ; les variables dominantes sont liées au contenu de l'entretien (mentions de difficultés cumulées, freins périphériques), au métier recherché et au département — cohérent avec une décision fondée sur des critères directement liés à l'employabilité.
- **Découverte importante** : malgré le retrait des variables sensibles, un écart résiduel de traitement subsiste par sous-groupe. La tranche d'âge 26-45 ans présente un taux d'erreur critique 2→0 de 21,4 %, contre 7,8 % en moyenne sur l'ensemble du holdout — signal net bien que reposant sur un nombre restreint de cas réels de classe 2 dans ce sous-groupe (14 cas), à confirmer par un test statistique formel avant de conclure à un biais structurel.
- Un second signal, de nature différente, est observé sur le profil « sans diplôme » : taux de fausses alertes (0→2) de 55,6 %, très supérieur aux autres profils de diplôme, alors que le taux d'erreur grave y est nul. Les deux signaux sont documentés séparément (l'un touchant la sécurité de l'usager, l'autre la charge opérationnelle des conseillers) plutôt que résumés en un unique constat de biais.
- Cette observation confirme un point de méthode important : retirer une variable sensible du modèle ne garantit pas la disparition d'un écart de traitement corrélé (discrimination indirecte via des variables proxy comme le métier recherché ou le département) — le suivi de cet écart doit donc être maintenu en continu plutôt que traité comme clos après le seul retrait de la nationalité et de l'âge.

> 🔎 **Note de relecture** — Le signal 26–45 ans repose sur 14 cas. À titre illustratif, si ~3 des 14 cas sont des 2→0 contre ~4 sur les 76 autres cas de classe 2, un test exact de Fisher donne p ≈ 0,07 (non significatif à 5 %) et l'intervalle de Wilson de 21,4 % sur 14 cas est d'environ [8 % ; 48 %]. Le test formel annoncé reste à réaliser sur les effectifs réels (annexe A4, écart n°9).

### ✅ Prochaines étapes
- Concevoir le mécanisme de seuil de rejet / abstention (§7.2.2) destiné à limiter le risque résiduel d'erreur critique sur les cas les plus incertains.
- Vérifier la méthode de calibrage de ce seuil pour éviter toute contamination du jeu de test final.

---

<a id="jour-10"></a>
## 📅 Journal de bord – Jour 10 (03/09/2026)

### 🔧 Étapes / Actions réalisées
- Première implémentation d'un mécanisme de seuil de rejet (§7.2.2) : abstention du modèle lorsque la probabilité maximale prédite reste sous un seuil de confiance, avec une règle complémentaire dédiée à la classe 2 (escalade humaine si une prédiction « classe 0 » s'accompagne d'une probabilité non négligeable de classe 2).
- **Correction méthodologique** : le premier balayage de seuils avait été réalisé directement sur le jeu de test final (holdout), ce qui contamine la mesure de performance annoncée — le seuil ne peut pas être choisi sur les mêmes données que celles utilisées pour rapporter le résultat final. Le protocole a été revu en trois temps : un jeu de calibration dédié (distinct du train d'entraînement et du holdout) sert à calibrer les probabilités (`CalibratedClassifierCV`) et à balayer les seuils ; le holdout n'est utilisé qu'une seule fois, en toute fin, pour confirmer le seuil retenu.
- Balayage du compromis abstention / fiabilité sur le jeu de calibration corrigé, avec deux contraintes croisées : un taux d'erreur critique résiduel maximal et un taux d'abstention maximal (capacité de revue humaine).

### 📝 Observations / Difficultés
- Aucun seuil unique ne permet de satisfaire simultanément un objectif initial de 5 % d'erreur critique résiduelle et un taux d'abstention inférieur à 30 % : atteindre 5 % exigerait d'écarter plus de 64 % des dossiers, ce qui rendrait le délai de traitement humain intenable. L'objectif de 5 % est donc documenté comme une hypothèse de départ non validée par un porteur métier, plutôt que comme une contrainte intangible, et révisé à 8 % sur la base du point d'équilibre observé empiriquement (~30 % d'abstention).
- La distribution des probabilités prédites sur les erreurs graves montre que celles-ci se concentrent sur des probabilités de classe 2 très basses : le modèle n'hésite pas sur ces cas, il est confiant et se trompe. Un simple seuil de confiance globale ne peut donc pas les détecter ; c'est ce qui justifie la règle complémentaire spécifique à la classe 2, dont l'utilité est ainsi confirmée empiriquement plutôt que supposée.
- La règle spécifique à la classe 2 réglée pour capter la majorité des erreurs graves entraîne un taux d'escalade élevé sur les dossiers prédits « classe 0 » (environ deux tiers) — compromis opérationnel important à faire valider explicitement par le porteur métier avant toute mise en production, plutôt qu'à présenter comme acquis.

> 🔎 **Note de relecture** — L'objectif de 8 % retenu ici est devenu 8,5 % dans le tableau de §9.1 (« seuil métier retenu en §7.2.2 »). Une seule valeur doit faire foi (annexe A4, écart n°2). Par ailleurs, la calibration décrite ici (`CalibratedClassifierCV`) concerne le protocole de choix du seuil ; §9.4 indique que l'artefact actuellement servi est un pipeline non calibré, ce qui remet en cause la transposition des seuils 0,55 / 0,04 (écart n°3).

### ✅ Prochaines étapes
- Documenter l'architecture d'industrialisation (§8) : services API, contrat d'interface, gestion des erreurs.
- Vérifier la cohérence entre le contrat d'API et les variables réellement utilisées par le modèle (notamment l'absence d'âge et de nationalité).

---

<a id="jour-11"></a>
## 📅 Journal de bord – Jour 11 (04/09/2026)

### 🔧 Étapes / Actions réalisées
- Documentation de l'architecture d'industrialisation (§8) : service `model` (FastAPI, prédiction unitaire, réentraînement protégé par jeton), service `backend` (orchestration, historique, feedback conseiller), frontend Nginx, documentation Swagger, catalogue des codes d'erreur (403/404/409/422/500/502/503).
- Revue de cohérence entre le schéma de l'API et le scénario retenu : suppression du champ `age` du schéma de validation de la route `/predict`, ce champ n'étant plus utilisé par le modèle depuis le retrait des variables sensibles (§6). Vérification qu'aucune autre trace de l'âge ne subsiste dans le stockage de l'historique.
- Rédaction du plan de suivi en production (§9) : métriques de monitoring (§9.1) reliées explicitement aux constats des sections précédentes (taux d'erreur critique, écart par sous-groupe issu de l'audit §7.2.1, dérive des variables les plus influentes selon SHAP), boucle de rétroaction (§9.2) et plan de réentraînement (§9.3).
- Mise en place d'un pipeline de réentraînement automatisé (GitHub Actions) déclenché par volume de feedbacks et par calendrier, avec comparaison challenger/champion avant toute promotion.

### 📝 Observations / Difficultés
- Le suivi de l'écart par sous-groupe (§9.1) identifié en §7.2.1 ne peut pas être reconduit à l'identique en production : l'interface ne collecte plus l'âge ni la nationalité, conformément au principe de minimisation des données. Ce suivi continu est donc remplacé par des campagnes d'audit périodiques, avec une collecte dédiée et temporaire, distincte du flux de prédiction courant.
- Le premier pipeline de réentraînement reposait sur une connexion SSH vers la machine de déploiement, nécessitant plusieurs secrets sensibles (hôte, utilisateur, clé privée). Ce mécanisme a été remplacé par un runner GitHub auto-hébergé directement sur la machine de déploiement, supprimant l'exposition d'un port entrant et réduisant le nombre de secrets à gérer.
- **Point de vigilance corrigé** : la première version du mécanisme de promotion écrasait directement l'artefact du modèle en production sans sauvegarde préalable, rendant tout retour en arrière impossible en cas d'anomalie détectée après coup. Un mécanisme de sauvegarde systématique (modèle + configuration) avant chaque promotion a été ajouté, ainsi qu'une commande de rollback manuel et une restauration automatique en cas d'échec du redémarrage du service.

### ✅ Bilan et prochaines étapes
- L'ensemble des parties du projet (cadrage, données, EDA, préparation, modélisation, arbitrage éthique/performance/robustesse, explicabilité, industrialisation, suivi en production) est désormais couvert et documenté.
- Le modèle retenu (XGBoost, scénario sans variables sensibles) est accompagné d'un mécanisme d'abstention calibré correctement, d'un contrat d'API cohérent avec les choix éthiques du projet, et d'un pipeline de réentraînement traçable et réversible.
- Limites assumées à mentionner en soutenance : mesures de coût opérationnel non répétées statistiquement, écart résiduel par sous-groupe non totalement expliqué et à surveiller, ratio de pondération entre les deux types d'erreur et seuil d'abstention encore fondés sur des hypothèses de travail à faire valider par un porteur métier plutôt que sur une calibration métier définitive.
- Prochaine étape : préparation du support de soutenance (synthèse en langage métier, sélection des visuels clés, répétition du pitch en trois messages).

> 🔎 **Note de relecture** — La formule « mécanisme d'abstention calibré correctement » est à nuancer : les seuils ont été fixés avec des probabilités calibrées, alors que l'artefact servi ne l'est pas (§9.4). Formulation suggérée : « seuils d'abstention déterminés sur un jeu de calibration dédié ; leur pertinence sur l'artefact servi reste à confirmer après calibration ».

---

> **Note de chronologie** — Les journées 12 à 16 regroupent des jalons visibles dans le code et la documentation du dépôt. Les dates exactes de réalisation ne sont pas toutes traçables ici et sont indiquées *(date à confirmer)*.

<a id="jour-12"></a>
## 📅 Journal de bord – Jour 12 *(date à confirmer)* : intégration de l'application et boucle de retour

### 🔧 Étapes / Actions réalisées
- Consolidation de l'architecture exécutable avec Docker Compose : interface web (Nginx, port 8088), backend FastAPI (8001), service modèle FastAPI (8000), Prometheus, Grafana et MLflow.
- Vérification du parcours de prédiction : le backend transmet les entrées au modèle, renvoie la classe et ses trois probabilités, puis conserve le résultat et les éléments utiles à l'historique dans SQLite.
- Mise en place du retour conseiller (`/feedback`) : un feedback associe le résultat observé au `request_id` de la prédiction et vérifie la cohérence de la classe prédite avant stockage. Un rejeu identique est idempotent ; un feedback contradictoire est refusé (409), un `request_id` inconnu aussi (404).
- Documentation des routes et des codes d'erreur (200/201, 403, 404, 409, 422, 500, 502, 503), avec une commande `pytest` ciblée par code d'erreur.

### 📝 Observations / Difficultés
- La prédiction est une aide à la décision, pas une décision automatique : le résultat observé doit être renseigné séparément pour alimenter l'évaluation ultérieure.
- Les identifiants (`usager_id`, `session_id`, `request_id`) et les informations conservées dans l'historique exigent une attention particulière au regard de la minimisation des données : durée de conservation et base légale restent à documenter.
- `/health` du backend est une simple liveness : il reste « UP » même si le modèle est injoignable. Une route de readiness vérifiant le modèle éviterait qu'un tableau de bord vert masque une panne fonctionnelle.
- Les routes `/feedback`, `/history` et `/score` ne décrivent pas d'authentification : tant qu'elles ne sont exposées qu'en réseau interne, le risque est limité, mais un feedback falsifié pourrait empoisonner le réentraînement.

### ✅ Prochaines étapes
- Vérifier les contrôles de santé, les erreurs d'API et la persistance de l'historique sur un parcours complet.
- Contrôler que les retours enregistrés sont exploitables et suffisamment complets (maturité, unicité, couverture) avant de les utiliser pour l'entraînement.
- Ajouter des tests pour les codes 500, 502 et 503 (mock du modèle injoignable ou en erreur).

---

<a id="jour-13"></a>
## 📅 Journal de bord – Jour 13 *(date à confirmer)* : observabilité et analyse de dérive

### 🔧 Étapes / Actions réalisées
- Connexion des métriques des services à Prometheus et du dashboard Grafana à cette source de données : disponibilité, requêtes, erreurs 5xx, latence, répartition des classes prédites, confiance et taux de revue humaine.
- Ajout d'une analyse batch (`scripts/drift_analysis.py`) entre une période de référence et une période courante : PSI, tests KS et χ² sur les variables d'entrée, PSI/KS sur les probabilités `proba_0/1/2` quand elles sont disponibles, et diagnostic de concept drift si les labels existent.
- Publication du rapport JSON et des métriques Prometheus `cisia_drift_*` et `cisia_confidence_drift_*`, exposées par le backend sur `/drift-metrics`, avec un niveau de sévérité (0 aucun, 1 surveillance, 2 forte).
- Écriture de la logique de triangulation : un signal statistique doit être croisé avec le contexte, la qualité des données et, lorsque les labels sont disponibles, la performance du modèle.

### 📝 Observations / Difficultés
- Repères PSI : < 0,10 stable, 0,10–0,25 à surveiller, ≥ 0,25 forte dérive à investiguer. Ce sont des heuristiques issues du scoring de crédit, pas une preuve de dégradation ni une décision automatique de réentraînement.
- Le drift affiché est batch : il dépend des fichiers de référence et de la fenêtre courante, et ne se recalcule pas à chaque prédiction saisie dans l'interface. L'exécution quotidienne et les notifications ne sont pas encore configurées.
- **Résultat du dashboard (mode démonstration, données simulées)** : PSI de `code_insee_commune` = 7,26 (sévérité 2, statut « À INVESTIGUER »), alors que les autres PSI sont faibles (ancienneté 0,020 ; ROME 0,058 ; diplôme 0,009) et que les p-values KS/χ² sont toutes > 0,05 (dont χ² sur `code_insee_commune` : 0,694). Les PSI des probabilités restent < 0,02.
- **Lecture critique** : PSI très élevé et χ² non significatif se contredisent. L'hypothèse la plus plausible est un artefact du PSI catégoriel sur une variable à très haute cardinalité (2 407 modalités pour ~2 500 lignes, cf. J2) : les modalités absentes d'un jeu sont plafonnées à 1e-6, ce qui fait exploser le logarithme. À vérifier dans les données.
- **Couverture à revoir** : le modèle utilise `departement` (J5), pas `code_insee_commune`, que J2 avait jugé inutilisable tel quel. Le script surveille la mauvaise variable géographique et ne surveille ni `departement` ni `synthese_entretien`, pourtant parmi les variables les plus influentes selon SHAP (J9).

### ✅ Prochaines étapes
- Remplacer `code_insee_commune` par `departement` dans le suivi de dérive, ou regrouper les modalités rares avant de calculer le PSI.
- Ajouter un suivi de dérive pour `synthese_entretien` (longueur, vocabulaire, distribution des scores TF-IDF) et pour `departement`.
- Exposer les seuils `psi_watch = 0,10` et `psi_critical = 0,25` dans le rapport (aujourd'hui `psi_warning = 0,25` côté JSON et 0,10 codé en dur côté Prometheus).
- Définir un effectif minimal pour la fenêtre courante (par exemple ≥ 1 000 lignes) et planifier l'exécution quotidienne avec une alerte de péremption du rapport.

---

<a id="jour-14"></a>
## 📅 Journal de bord – Jour 14 *(date à confirmer)* : calibration et garde-fou d'évaluation

### 🔧 Étapes / Actions réalisées
- Formalisation de mesures de calibration sur les probabilités et les labels observés : tableau de fiabilité, ECE et comparaison de l'ECE courant à la période de référence (`src/calibration.py`). Le Brier score et le suivi mensuel automatisé ne sont pas encore implémentés.
- Mise en place d'un diagnostic qui distingue dérive des entrées, évolution de la relation aux labels et dérive de calibration, puis formule une recommandation proportionnée (`src/recommendations.py`) sans lancer de réentraînement.
- Stabilisation de l'évaluation de release (`scripts/evaluate_model.py`) autour d'un jeu de référence gelé et d'un golden run tracé dans MLflow. Le script compare F1-macro, F1 et recall de la classe 2, ainsi que ROC AUC OVR, et retourne un code de sortie 1 si les seuils sont franchis (mode `--degrade` pour tester volontairement le chemin d'échec).
- Détermination des planchers effectifs de la gate, calculés comme `max(minimum absolu, baseline − baisse maximale)` :

| Métrique | Baseline | Minimum absolu | Baisse max. | **Plancher effectif** | Règle active |
|---|---|---|---|---|---|
| F1-macro | 0,6455 | 0,60 | 0,04 | **0,6055** | relative |
| F1 classe 2 | 0,5395 | 0,50 | 0,05 | **0,50** | absolue |
| Recall classe 2 | 0,6444 | 0,59 | 0,06 | **0,59** | absolue |
| ROC AUC OVR macro | 0,8153 | 0,70 | 0,04 | **0,7753** | relative |

### 📝 Observations / Difficultés
- La calibration ne peut être évaluée correctement qu'avec des labels suffisamment mûrs ; un déplacement des probabilités seul ne remplace pas l'ECE.
- Une baseline doit être mesurée sur le même jeu de référence que le modèle évalué. Comparer des populations ou des jeux différents confondrait évolution du modèle et différence d'échantillonnage.
- **Bruit d'estimation** : avec ~90 cas de classe 2 dans le jeu gelé, le recall (0,6444 = 58/90) a un écart-type d'environ 0,05 et un intervalle de Wilson à 95 % d'environ [0,54 ; 0,74]. Une marge de 0,054 au-dessus du plancher de 0,59 correspond à un seul écart-type : une baisse de cette taille ne se distingue pas du hasard d'échantillonnage.
- Le gate actuel n'inclut ni la précision de la classe 2 (≈ 0,46, déduite de F1 et recall), ni la PR AUC de la classe 2 : le recall peut être conservé en sur-prédisant la classe 2. Le ROC AUC macro masque par ailleurs la classe prioritaire.
- Les recommandations de drift restent des heuristiques de diagnostic : elles ne remplacent ni la revue humaine ni la gate de qualité. Le gate de release contrôle la non-régression du modèle et du code ; il ne surveille pas le modèle en service.

### ✅ Prochaines étapes
- Vérifier que le jeu de référence contient les trois classes et qu'il reste figé entre releases (versionnage, revue semestrielle).
- Tester le chemin CI nominal et le chemin dégradé afin de confirmer que la gate bloque bien une régression.
- Ajouter `precision_classe_2` et `pr_auc_classe_2` à la gate, et un bootstrap apparié (5 000 tirages) pour comparer candidat et production sur le même jeu.
- Porter l'effectif de la classe 2 du jeu gelé vers ~400 cas si possible, afin de resserrer l'écart-type du recall autour de 0,024.

---

<a id="jour-15"></a>
## 📅 Journal de bord – Jour 15 *(date à confirmer)* : réentraînement contrôlé et promotion

### 🔧 Étapes / Actions réalisées
- Mise en place d'un workflow GitHub Actions exécutable manuellement ou toutes les 6 heures. Cette planification ne fait que **vérifier** les conditions de réentraînement : un candidat n'est entraîné que si 200 feedbacks validés et non consommés sont disponibles (valeur par défaut, paramétrable). Sinon, le statut `skipped_low_volume` est enregistré et les feedbacks sont conservés.
- Préparation d'un candidat à partir des feedbacks et des données historiques (`scripts/retrain.py`), avec exclusion du jeu de référence de l'entraînement et contrôle des recouvrements avec celui-ci.
- Séparation explicite entre création du candidat et promotion : comparaison au modèle de production sur le même jeu de référence, planchers de qualité, tolérances de régression et exigence d'un gain sur F1-macro ou recall de classe 2.
- Traçabilité de la décision et des artefacts (métriques, paramètres, décision) dans MLflow et le journal des décisions, avec sauvegarde de la configuration et du modèle promu pour permettre un rollback manuel.
- **État observé dans le dashboard (mode démonstration)** : dernière tentative `skipped_low_volume` avec 4 nouveaux feedbacks pour un seuil de 200 (candidat entraîné : 0/1) ; gate d'évaluation locale en `PASS` ; modèle déployé `v1.0.0 – multimodal_ethique`.

### 📝 Observations / Difficultés
- Atteindre le seuil de feedback déclenche une tentative d'entraînement, pas une promotion automatique : le candidat doit passer la gate de qualité. Les alertes de drift ou de performance ne déclenchent jamais de réentraînement ; elles ouvrent une investigation.
- Des étiquettes erronées, des entrées incomplètes ou un échantillon trop peu représentatif peuvent détériorer le candidat ; la validation des feedbacks reste essentielle. Les feedbacks sont facultatifs côté conseiller : l'échantillon n'est donc pas supposé représentatif, et les labels peuvent être influencés par la prédiction affichée.
- Le rollback et la conservation des volumes sont des exigences opérationnelles : une correction de modèle ne doit pas effacer l'historique de production. L'activation d'un modèle promu dans l'API et le rollback restent des étapes manuelles.
- **Critique de la règle de promotion** : exiger un « gain » de F1-macro ou de recall classe 2 est indiscernable du bruit avec ~90 cas, et empêche de promouvoir un modèle équivalent entraîné sur des données plus récentes, ce qui est pourtant l'objectif d'un réentraînement face à une dérive. Une règle de non-infériorité (borne basse de l'IC à 95 % de la différence candidat − production au-dessus d'une tolérance) est plus adaptée.
- **Logique volume vs besoin** : si une dégradation est confirmée mais que moins de 200 feedbacks sont disponibles, le workflow répond `skipped_low_volume`. Une voie manuelle, tracée dans le journal des décisions, doit permettre d'abaisser le seuil après investigation (à trancher).
- Le seuil de 200 est un total : il peut ne contenir qu'une poignée de cas de classe 2. Un effectif minimal par classe est à ajouter.

### ✅ Bilan et prochaines étapes
- Le projet couvre désormais le cycle allant de l'analyse métier et de la modélisation jusqu'au scoring, au suivi, à l'évaluation continue et à la préparation contrôlée d'un candidat.
- Les décisions de déploiement restent conditionnées par les tests, l'évaluation sur référence et la revue des signaux de dérive ; un indicateur isolé ne suffit pas.
- À compléter : dates réellement travaillées et résultats des dernières exécutions CI, drift et retraining sur données réelles (les valeurs ci-dessus proviennent d'un tableau de bord en mode démonstration).

---

<a id="jour-16"></a>
## 📅 Journal de bord – Jour 16 *(date à confirmer)* : revue de cohérence et durcissement de la documentation

### 🔧 Étapes / Actions réalisées
- **§9.2 Boucle de rétroaction** : réécriture pour supprimer la redondance avec §9.3 et ajout de trois règles d'éligibilité (maturité minimale, unicité par dossier ou personne, suivi du taux de couverture), ainsi que de la distinction « consommé pour l'entraînement » / « disponible pour la mesure ».
- **§9.5 Synthèse** : restructuration en cadence de surveillance par couche, déclencheurs de réentraînement (périodique guidé par le volume, ou manuel après investigation) et plan de remédiation par situation avec responsable.
- **§8.6 Synthèse industrialisation** : tableau de pile resserré ; le proxy `/train` du backend n'y figure plus.
- **Preuves de test** : liste de commandes `pytest` reliant chaque code d'erreur documenté (403, 404, 409, 422) à un test exécutable.
- **Revue d'ensemble** de §8.6, §9.1 à §9.5 et de la documentation API : relevé des écarts consignés en annexe A4.

### 📝 Observations / Difficultés
- Les correctifs ne sont pas encore tous reportés : §9.3 conserve la règle « amélioration du F1-macro ou du recall » et ne traite ni la précision de la classe 2, ni le test apparié, ni la calibration, ni l'effectif par classe ; §9.1 garde un seuil de concept drift à 0,03 (plus strict que la gate de release à 0,04 alors qu'il est mesuré sur des fenêtres plus bruitées) ; §9.4 contient encore un premier tableau de gabarit (`💡/⚠️`, « C8 N3 ») en doublon.
- Les codes 500, 502 et 503 sont documentés sans test référencé ; le sélecteur `-k` de pytest fait une correspondance par sous-chaîne (préférer `fichier::test_nom`) ; la règle « minimum 10 dossiers » de `/train` est visible dans les noms de tests mais absente de la documentation API.
- La documentation API cite encore « âge hors bornes » (422) alors que le champ `age` a été retiré du schéma en J11 ; un usager inconnu y renvoie 503 (service indisponible) au lieu d'un 404 ou 422.
- Sécurité : `/train` n'est contrôlé que si `TRAIN_API_TOKEN` est défini ; `/metrics` et `/drift-metrics` sont absents de Swagger mais ne sont pas « protégés » pour autant (à limiter au réseau de supervision).

### ✅ Prochaines étapes
- Appliquer à §9.3 les critères de promotion par non-infériorité, la précision de la classe 2, la stabilité du taux de revue humaine, la calibration et l'effectif minimal par classe.
- Mettre à jour §9.1 (précision et PR AUC classe 2, concept drift aligné sur 0,04, paliers PSI à deux niveaux) et supprimer le gabarit de §9.4.
- Ajouter à §8.6 la ligne « Réentraînement » et la section « Points encore ouverts ».
- Suivre le plan d'action ordonné de l'annexe A6.

---